<a href="https://colab.research.google.com/github/kartik2006-del/biopapers-rag/blob/main/BioPapers_RAG_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metadata Parsing

In [1]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET




In [2]:
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": '"CRISPR"[Title/Abstract] AND "gene editing"[Title/Abstract]',
    "retmax": 5,
    "retmode": "json"
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)
print("PMIDs:", response.json()["esearchresult"]["idlist"])

Status code: 200
PMIDs: ['42906700', '42904570', '42902180', '42899764', '42899205']


In [3]:
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

fetch_params = {
    "db": "pubmed",
    "id": ",".join(response.json()["esearchresult"]["idlist"]),
    "rettype": "abstract",
    "retmode": "xml"
}

fetch_response = requests.get(
    fetch_url,
    params=fetch_params,
    timeout=30
)

print("Status code:", fetch_response.status_code)
print(fetch_response.text[:1000])

Status code: 200
<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
<PubmedArticle><MedlineCitation Status="Publisher" Owner="NLM"><PMID Version="1">42906700</PMID><DateRevised><Year>2026</Year><Month>10</Month><Day>22</Day></DateRevised><Article PubModel="Print-Electronic"><Journal><ISSN IssnType="Electronic">1873-2259</ISSN><JournalIssue CitedMedium="Internet"><PubDate><Year>2026</Year><Month>Oct</Month><Day>21</Day></PubDate></JournalIssue><Title>Plant science : an international journal of experimental plant biology</Title><ISOAbbreviation>Plant Sci</ISOAbbreviation></Journal><ArticleTitle>The conserved function of bHLH121 and overexpression of N-terminal bHLH121 fragment suppresses iron deficiency response in Arabidopsis.</ArticleTitle><Pagination><StartPage>113540</StartPage><MedlinePgn>113540</MedlinePgn></Pagination><ELocationID EIdType="doi" Va

In [4]:
root = ET.fromstring(fetch_response.content)

article = root.find(".//PubmedArticle")

title = article.findtext(
    ".//ArticleTitle",
    default="Title not found"
)

print("First paper title:")
print(title)

First paper title:
The conserved function of bHLH121 and overexpression of N-terminal bHLH121 fragment suppresses iron deficiency response in Arabidopsis.


In [5]:
articles = root.findall(".//PubmedArticle")

for article in articles:
    pmid = article.findtext(".//PMID", default="N/A")
    title = article.findtext(".//ArticleTitle", default="N/A")

    print("PMID:", pmid)
    print("Title:", title)
    print("-" * 80)

PMID: 42906700
Title: The conserved function of bHLH121 and overexpression of N-terminal bHLH121 fragment suppresses iron deficiency response in Arabidopsis.
--------------------------------------------------------------------------------
PMID: 42904570
Title: Synthetic Biology Toolbox for Lactiplantibacillus plantarum: Engineering Next-Generation Probiotic Cell Factories.
--------------------------------------------------------------------------------
PMID: 42902180
Title: Periplocin targets low density lipoprotein receptor-related protein 4 and protects against osteoarthritis.
--------------------------------------------------------------------------------
PMID: 42899764
Title: Nano Delivery Systems for In Vivo CRISPR/Cas-Based Gene Therapy: From Vectors to Clinical Outcomes.
--------------------------------------------------------------------------------
PMID: 42899205
Title: Genome editing beyond transgenes: DNA-free technologies in crop improvement.
-------------------------------

In [6]:

records = []

for article in articles:
    pmid = article.findtext(".//PMID", default="N/A")
    title = article.findtext(".//ArticleTitle", default="N/A")

    journal = article.findtext(
        ".//Journal/Title",
        default="N/A"
    )

    abstract_parts = [
        part.itertext()
        for part in article.findall(".//Abstract/AbstractText")
    ]

    abstract = " ".join(
        text
        for part in abstract_parts
        for text in part
    )

    records.append({
        "PMID": pmid,
        "Title": title,
        "Journal": journal,
        "Abstract": abstract,
        "Source_URL": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
    })

print("Records parsed:", len(records))
print("First record:", records[0])


Records parsed: 5
First record: {'PMID': '42906700', 'Title': 'The conserved function of bHLH121 and overexpression of N-terminal bHLH121 fragment suppresses iron deficiency response in Arabidopsis.', 'Journal': 'Plant science : an international journal of experimental plant biology', 'Abstract': 'Iron (Fe) homeostasis is fundamental to plant growth and development and is centrally regulated by a well-conserved network of basic helix-loop-helix (bHLH) transcription factors. In Arabidopsis thaliana, bHLH121 regulates the expression of FER-like iron deficiency-induced transcription factor (FIT) and clade Ib bHLH38/39/100/101 genes, among others. However, its functional conservation remains unexplored, and some contradictory results have been reported regarding the effects of bHLH121 overexpression. Here, we investigated the evolutionary conservation of bHLH121 through complementary tests in the Arabidopsis bhlh121 mutant created by CRISPR/Cas9 gene editing. We also investigated the effec

In [7]:
df = pd.DataFrame(records)

print("Dataset shape:", df.shape)
display(df[["PMID", "Title", "Journal"]])

Dataset shape: (5, 5)


,PMID,Title,Journal
0,42906700,The conserved function of bHLH121 and overexpr...,Plant science : an international journal of ex...
1,42904570,Synthetic Biology Toolbox for Lactiplantibacil...,Biotechnology journal
2,42902180,Periplocin targets low density lipoprotein rec...,Cellular & molecular biology letters
3,42899764,Nano Delivery Systems for In Vivo CRISPR/Cas-B...,ACS nano
4,42899205,Genome editing beyond transgenes: DNA-free tec...,Frontiers in plant science


In [8]:

for article in articles:
    print("PMID:", article.findtext(".//PMID", default="N/A"))

    authors = []
    for author in article.findall("./MedlineCitation/Article/AuthorList/Author"):
        fore = author.findtext("ForeName", default="")
        last = author.findtext("LastName", default="")
        name = f"{fore} {last}".strip()

        if name:
            authors.append(name)

    print("Authors:", ", ".join(authors) if authors else "N/A")

    pub_date = article.find("./MedlineCitation/Article/Journal/JournalIssue/PubDate")

    year = "N/A"
    if pub_date is not None:
        year = pub_date.findtext("Year", default="N/A")

        if year == "N/A":
            medline_date = pub_date.findtext("MedlineDate", default="")
            if medline_date:
                year = medline_date[:4]

    print("Publication year:", year)
    print("-" * 60)


PMID: 42906700
Authors: Peijun Zhou, Chuanfa Liu, Yilin Pan, Yuchen Fei, Renfang Shen, Ruonan Wang, Ping Lan
Publication year: 2026
------------------------------------------------------------
PMID: 42904570
Authors: Peiyun Su, Yulin Ma, Qian Nie, Tingxing Yang, Tao Wei
Publication year: 2026
------------------------------------------------------------
PMID: 42902180
Authors: Jincheng Liu, Qingyang Fu, Chunlin Li, Kun Gao, Mengsi Sun, Mingyu Xu, Junfei Chen, Xu Zhai, Wencan Zhang, Le Li, Wanlong Xu, Yang Liu, Xinhui Wu, Haipeng Si, Kaidi Wang
Publication year: 2026
------------------------------------------------------------
PMID: 42899764
Authors: Ruhan Chen, Haimei Zhou, Qinjie Wu, Changyang Gong
Publication year: 2026
------------------------------------------------------------
PMID: 42899205
Authors: Roshini Senthilkumar, Rakshana Palaniswamy, Sakthi Ambothi Rathnasamy, Veera Ranjani Rajagopalan, Kokiladevi Eswaran, Radhamani Sengodan, Ashok Kumar Govindasamy, Raveendran Muthurajan

In [9]:

records = []

for article in articles:
    pmid = article.findtext(".//PMID", default="N/A")
    title = article.findtext(".//ArticleTitle", default="N/A")
    journal = article.findtext(
        "./MedlineCitation/Article/Journal/Title",
        default="N/A"
    )

    abstract_parts = [
        "".join(part.itertext())
        for part in article.findall(
            "./MedlineCitation/Article/Abstract/AbstractText"
        )
    ]
    abstract = " ".join(abstract_parts)

    pub_date = article.find(
        "./MedlineCitation/Article/Journal/JournalIssue/PubDate"
    )
    year = "N/A"

    if pub_date is not None:
        year = pub_date.findtext("Year", default="N/A")
        if year == "N/A":
            medline_date = pub_date.findtext("MedlineDate", default="")
            if medline_date:
                year = medline_date[:4]

    authors = []
    for author in article.findall(
        "./MedlineCitation/Article/AuthorList/Author"
    ):
        fore = author.findtext("ForeName", default="")
        last = author.findtext("LastName", default="")
        name = f"{fore} {last}".strip()
        if name:
            authors.append(name)

    records.append({
        "PMID": pmid,
        "Title": title,
        "Year": year,
        "Authors": ", ".join(authors),
        "Journal": journal,
        "Abstract": abstract,
        "Source_URL": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
    })

df = pd.DataFrame(records)

print("Dataset shape:", df.shape)
display(df[["PMID", "Year", "Title", "Authors", "Journal"]])


Dataset shape: (5, 7)


,PMID,Year,Title,Authors,Journal
0,42906700,2026,The conserved function of bHLH121 and overexpr...,"Peijun Zhou, Chuanfa Liu, Yilin Pan, Yuchen Fe...",Plant science : an international journal of ex...
1,42904570,2026,Synthetic Biology Toolbox for Lactiplantibacil...,"Peiyun Su, Yulin Ma, Qian Nie, Tingxing Yang, ...",Biotechnology journal
2,42902180,2026,Periplocin targets low density lipoprotein rec...,"Jincheng Liu, Qingyang Fu, Chunlin Li, Kun Gao...",Cellular & molecular biology letters
3,42899764,2026,Nano Delivery Systems for In Vivo CRISPR/Cas-B...,"Ruhan Chen, Haimei Zhou, Qinjie Wu, Changyang ...",ACS nano
4,42899205,2026,Genome editing beyond transgenes: DNA-free tec...,"Roshini Senthilkumar, Rakshana Palaniswamy, Sa...",Frontiers in plant science
